In [ ]:
with first_view as (

    select
        magnit_id,
        min(calc_date::date) as first_view_date
    from mobapp_act
    where metric_id = 1
    group by magnit_id

),

subscr_min as (

    select
        contact_id,
        min(subscr_date_act::date) as first_subscr_date
    from subscr_status
    group by contact_id

),

base as (

    select
        cc.client_id,
        cc.magnit_id,
        cc.campaigns_cnt,
        fv.first_view_date,
        sm.first_subscr_date,

        case
            when fv.first_view_date is not null then 1
            else 0
        end as saw_offer_flg,

        case
            when fv.first_view_date is not null
             and (
                    sm.first_subscr_date is null
                    or sm.first_subscr_date < fv.first_view_date
                 )
            then 1
            else 0
        end as not_subscr_flg

    from client_cohorts cc

    left join first_view fv
        on cc.magnit_id = fv.magnit_id

    left join subscr_min sm
        on cc.client_id = sm.contact_id

)

select
    campaigns_cnt,

    count(distinct client_id) as total_clients,

    count(distinct case
        when saw_offer_flg = 1 then client_id
    end) as saw_offer_clients,

    count(distinct case
        when not_subscr_flg = 1 then client_id
    end) as not_subscr_clients,

    round(
        count(distinct case when saw_offer_flg = 1 then client_id end) * 100.0
        / nullif(count(distinct client_id), 0),
        1
    ) as saw_offer_pct,

    round(
        count(distinct case when not_subscr_flg = 1 then client_id end) * 100.0
        / nullif(count(distinct case when saw_offer_flg = 1 then client_id end), 0),
        1
    ) as not_subscr_pct_from_viewers,

    round(
        count(distinct case when not_subscr_flg = 1 then client_id end) * 100.0
        / nullif(count(distinct client_id), 0),
        1
    ) as not_subscr_pct_total

from base

group by campaigns_cnt

order by campaigns_cnt;